# 第310章: DDIM反転入門

## 📋 この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] DDIMの決定的サンプリングとDDPMの確率的サンプリングの違いを説明できる
- [ ] DDIM Inversionの概念（画像→ノイズへの逆過程）を理解し、図解できる
- [ ] 簡易的なDDIM拡散モデルを実装し、正方向・逆方向の変換を実験できる
- [ ] DDIM Inversionが画像編集パイプラインでなぜ重要かを説明できる

## 🎯 前提知識

- ✅ Notebook 39-43（拡散モデル基礎 — ノイズスケジュール、DDPM）
- ✅ Notebook 300-309（潜在空間と画像変容の基礎）
- ✅ 拡散モデルの正過程と逆過程の概念

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★★☆（上級）
🎓 **カテゴリ**: 理論・実践

---

## 🌟 はじめに — Phase 3: 拡散モデルベースの画像変容へ

Phase 2ではGANの潜在空間を操作しましたが、GANには大きな課題がありました：
- エンコーダがない → 実画像の編集が困難
- モード崩壊のリスク

Phase 3では**拡散モデル**を使った画像変容を学びます。
拡散モデルの最大の利点は**DDIM Inversion**です。

### 🤔 DDIM Inversionとは？

> **実画像をノイズに変換し（Inversion）、途中で編集して、再び画像に戻す技術**

```
  GAN方式:                          拡散モデル方式（DDIM Inversion）:
  ────────                          ──────────────────────────────

  入力画像 → ？？？ → z              入力画像 ──→ DDIM Inversion ──→ xₜ（ノイズ画像）
       エンコーダがない！                               │
                                                       ▼ ここで編集！
                                    出力画像 ←── DDIM Sampling ←── xₜ'（編集後）
```

### GAN Inversionとの比較

| 特性 | GAN Inversion | DDIM Inversion |
|------|-------------|----------------|
| 方法 | 最適化で z を推定 | 決定的逆過程 |
| 速度 | 遅い（数百ステップの最適化） | 速い（20-50ステップ） |
| 再構成精度 | 近似的 | ほぼ完璧 |
| 編集の自由度 | 潜在空間の操作 | ノイズ/テキスト/U-Netの操作 |

### 📝 この章の構成

1. **DDPMとDDIMの違い** — 確率的 vs 決定的サンプリング
2. **DDIM Inversionの理論** — 逆過程の数学
3. **シンプルな拡散モデルの実装** — MNISTで学習
4. **DDIM Inversionの実装** — 画像→ノイズ→再構成
5. **Inversionの精度検証** — 元画像との比較

In [ ]:
# ============================================================
# 環境設定
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)

print(f"Device: {device}")
print("✅ 環境設定完了")

---

## 1. DDPMとDDIMの違い

### DDPM（確率的サンプリング）

DDPMの逆過程には**ランダムノイズ**が含まれます：

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t \mathbf{z}$$

🎲 `σₜz` のランダム項があるため、**同じxₜから毎回異なる画像が生成**されます。

### DDIM（決定的サンプリング）

DDIMはランダム項を**ゼロ**にします（η=0の場合）：

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \underbrace{\left(\frac{x_t - \sqrt{1-\bar{\alpha}_t}\epsilon_\theta(x_t,t)}{\sqrt{\bar{\alpha}_t}}\right)}_{\text{予測された } x_0} + \sqrt{1-\bar{\alpha}_{t-1}} \cdot \epsilon_\theta(x_t, t)$$

🎯 ランダム項がないため、**同じxₜからは常に同じ画像が生成**されます = **決定的**

### なぜ決定的であることが重要か？

```
  DDPM:  x_T ──(ランダム)──→ x₀    ... 逆も非決定的 → Inversion不可能
  DDIM:  x_T ──(決定的)───→ x₀    ... 逆も決定的 → Inversion可能！
```

---

## 2. DDIM Inversionの理論

DDIMが決定的なら、その**逆過程も決定的**です。

### DDIM Inversion（正方向：画像→ノイズ）

$$x_{t+1} = \sqrt{\bar{\alpha}_{t+1}} \left(\frac{x_t - \sqrt{1-\bar{\alpha}_t}\epsilon_\theta(x_t,t)}{\sqrt{\bar{\alpha}_t}}\right) + \sqrt{1-\bar{\alpha}_{t+1}} \cdot \epsilon_\theta(x_t, t)$$

これは、DDIMサンプリングの式を $x_{t-1}$ ではなく $x_{t+1}$ について解いたものです。

### パイプライン

```
  画像編集パイプライン:

  元画像 x₀ ──→ DDIM Inversion ──→ xₜ ──→ 編集 ──→ x'ₜ ──→ DDIM Sampling ──→ 編集後画像 x'₀
                  (画像→ノイズ)                              (ノイズ→画像)

  「編集」の例:
  ・テキストプロンプトを変更
  ・ノイズ空間での補間
  ・U-Netの特徴量を操作
```

In [ ]:
# ============================================================
# シンプルな拡散モデルの実装
# ============================================================

# ノイズスケジュール
def get_schedule(n_steps=100, beta_start=1e-4, beta_end=0.02):
    """線形ノイズスケジュール"""
    betas = np.linspace(beta_start, beta_end, n_steps)
    alphas = 1.0 - betas
    alpha_bars = np.cumprod(alphas)
    return {
        'betas': torch.tensor(betas, dtype=torch.float32),
        'alphas': torch.tensor(alphas, dtype=torch.float32),
        'alpha_bars': torch.tensor(alpha_bars, dtype=torch.float32),
    }

# ノイズ予測ネットワーク
class NoisePredictor(nn.Module):
    """タイムステップ条件付きノイズ予測器"""
    def __init__(self, dim=784, time_emb_dim=32):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.net = nn.Sequential(
            nn.Linear(dim + time_emb_dim, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, dim),
        )

    def forward(self, x, t):
        # t: (batch_size,) の整数
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / 100.0)
        x_t = torch.cat([x, t_emb], dim=-1)
        return self.net(x_t)

schedule = get_schedule(n_steps=100)
noise_pred = NoisePredictor().to(device)

print(f"✅ 拡散モデル定義完了")
print(f"   パラメータ数: {sum(p.numel() for p in noise_pred.parameters()):,}")
print(f"   タイムステップ数: 100")

In [ ]:
# ============================================================
# 拡散モデルの学習
# ============================================================

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

alpha_bars = schedule['alpha_bars'].to(device)
optimizer = optim.Adam(noise_pred.parameters(), lr=2e-4)

n_epochs = 15
n_steps = 100

print("拡散モデル学習中...")
for epoch in range(n_epochs):
    noise_pred.train()
    total_loss = 0
    for x, _ in train_loader:
        x = x.view(-1, 784).to(device)
        x = x * 2 - 1  # [0,1] → [-1,1]

        # ランダムなタイムステップ
        t = torch.randint(0, n_steps, (x.shape[0],)).to(device)
        ab = alpha_bars[t].unsqueeze(-1)

        # 正方向: x₀にノイズを加える
        noise = torch.randn_like(x)
        x_noisy = torch.sqrt(ab) * x + torch.sqrt(1 - ab) * noise

        # ノイズを予測
        noise_pred_out = noise_pred(x_noisy, t)
        loss = nn.functional.mse_loss(noise_pred_out, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 3 == 0:
        print(f"  Epoch {epoch+1}/{n_epochs} | Loss: {total_loss/len(train_loader):.6f}")

print("✅ 拡散モデル学習完了")

In [ ]:
# ============================================================
# DDIMサンプリング（ノイズ→画像）
# ============================================================

@torch.no_grad()
def ddim_sample(noise_pred_model, x_T, schedule, n_inference_steps=50):
    """DDIMサンプリング（決定的）"""
    alpha_bars = schedule['alpha_bars'].to(device)
    n_train_steps = len(alpha_bars)

    # サブシーケンインデックス
    step_indices = np.linspace(0, n_train_steps - 1, n_inference_steps + 1, dtype=int)

    x = x_T.clone()
    trajectory = [x.cpu().clone()]

    for i in range(n_inference_steps, 0, -1):
        t_curr = step_indices[i]
        t_prev = step_indices[i - 1]

        t_tensor = torch.full((x.shape[0],), t_curr, dtype=torch.long).to(device)
        eps = noise_pred_model(x, t_tensor)

        ab_curr = alpha_bars[t_curr]
        ab_prev = alpha_bars[t_prev]

        # DDIMサンプリング式（η=0、決定的）
        x0_pred = (x - torch.sqrt(1 - ab_curr) * eps) / torch.sqrt(ab_curr)
        x = torch.sqrt(ab_prev) * x0_pred + torch.sqrt(1 - ab_prev) * eps

        if i % (n_inference_steps // 10) == 0:
            trajectory.append(x.cpu().clone())

    trajectory.append(x.cpu().clone())
    return x, trajectory

# テスト: ランダムノイズから画像生成
noise_pred.eval()
x_T = torch.randn(16, 784).to(device)
x_gen, traj = ddim_sample(noise_pred, x_T, schedule, n_inference_steps=50)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(16):
    img = (x_gen[i].cpu().numpy().reshape(28, 28) + 1) / 2
    img = np.clip(img, 0, 1)
    axes[i // 8, i % 8].imshow(img, cmap='gray')
    axes[i // 8, i % 8].axis('off')

fig.suptitle('DDIMサンプリングで生成（ノイズ→画像）', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_310_01_ddim_sampling.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ DDIMサンプリング成功")

---

## 3. DDIM Inversionの実装

DDIMサンプリングの数式を**逆方向**に適用し、画像→ノイズの変換を行います。

これがDDIM Inversionの核心です。

In [ ]:
# ============================================================
# DDIM Inversion（画像→ノイズ）
# ============================================================

@torch.no_grad()
def ddim_inversion(noise_pred_model, x_0, schedule, n_inference_steps=50):
    """DDIM Inversion（決定的逆変換: 画像→ノイズ）"""
    alpha_bars = schedule['alpha_bars'].to(device)
    n_train_steps = len(alpha_bars)

    step_indices = np.linspace(0, n_train_steps - 1, n_inference_steps + 1, dtype=int)

    x = x_0.clone()
    trajectory = [x.cpu().clone()]

    for i in range(n_inference_steps):
        t_curr = step_indices[i]
        t_next = step_indices[i + 1]

        t_tensor = torch.full((x.shape[0],), t_curr, dtype=torch.long).to(device)
        eps = noise_pred_model(x, t_tensor)

        ab_curr = alpha_bars[t_curr]
        ab_next = alpha_bars[t_next]

        # DDIM Inversion式（逆方向）
        x0_pred = (x - torch.sqrt(1 - ab_curr) * eps) / torch.sqrt(ab_curr)
        x = torch.sqrt(ab_next) * x0_pred + torch.sqrt(1 - ab_next) * eps

        if (i + 1) % (n_inference_steps // 10) == 0:
            trajectory.append(x.cpu().clone())

    trajectory.append(x.cpu().clone())
    return x, trajectory

# テスト画像を取得
test_batch = next(iter(test_loader))[0][:8]
x_0 = (test_batch.view(-1, 784).to(device) * 2 - 1)  # [0,1] → [-1,1]

# Inversion: 画像→ノイズ
print("DDIM Inversion中...")
x_T_inv, inv_traj = ddim_inversion(noise_pred, x_0, schedule, n_inference_steps=50)
print(f"  Inversionされたノイズのノルム: {x_T_inv.norm(dim=-1).mean():.2f}")

# 再構成: ノイズ→画像
print("DDIMサンプリングで再構成中...")
x_recon, recon_traj = ddim_sample(noise_pred, x_T_inv, schedule, n_inference_steps=50)

print("✅ Inversion + 再構成完了")

In [ ]:
# ============================================================
# Inversion精度の検証: 元画像 vs 再構成画像
# ============================================================

fig, axes = plt.subplots(3, 8, figsize=(14, 5.5))

for i in range(8):
    # 元画像
    img_orig = (x_0[i].cpu().numpy().reshape(28, 28) + 1) / 2
    axes[0, i].imshow(np.clip(img_orig, 0, 1), cmap='gray')
    axes[0, i].axis('off')

    # Inversionノイズ
    img_noise = x_T_inv[i].cpu().numpy().reshape(28, 28)
    axes[1, i].imshow(img_noise, cmap='RdBu_r', vmin=-3, vmax=3)
    axes[1, i].axis('off')

    # 再構成
    img_recon = (x_recon[i].cpu().numpy().reshape(28, 28) + 1) / 2
    axes[2, i].imshow(np.clip(img_recon, 0, 1), cmap='gray')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('元画像', fontsize=12, rotation=0, labelpad=40)
axes[1, 0].set_ylabel('ノイズ xₜ', fontsize=12, rotation=0, labelpad=40)
axes[2, 0].set_ylabel('再構成', fontsize=12, rotation=0, labelpad=40)

# MSEを計算
mse = nn.functional.mse_loss(x_recon.cpu(), x_0.cpu()).item()
fig.suptitle(f'DDIM Inversion + 再構成の精度\n元画像→ノイズ→再構成 (MSE={mse:.6f})',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_310_02_inversion_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"💡 再構成MSE: {mse:.6f}")
print("   GANのInversion（最適化ベース）と比べて、非常に高精度な再構成が可能")

In [ ]:
# ============================================================
# Inversionプロセスの可視化
# 画像が段階的にノイズに変わっていく過程
# ============================================================

fig, axes = plt.subplots(2, len(inv_traj), figsize=(18, 4))

sample_idx = 0

for col, snapshot in enumerate(inv_traj):
    img = snapshot[sample_idx].numpy().reshape(28, 28)
    img_show = (img + 1) / 2

    axes[0, col].imshow(np.clip(img_show, 0, 1), cmap='gray')
    axes[0, col].axis('off')
    if col == 0: axes[0, col].set_ylabel('Inversion\n(→ノイズ)', fontsize=10, rotation=0, labelpad=50)

for col, snapshot in enumerate(reversed(recon_traj)):
    img = snapshot[sample_idx].numpy().reshape(28, 28)
    img_show = (img + 1) / 2

    axes[1, col].imshow(np.clip(img_show, 0, 1), cmap='gray')
    axes[1, col].axis('off')
    if col == 0: axes[1, col].set_ylabel('Sampling\n(→画像)', fontsize=10, rotation=0, labelpad=50)

fig.suptitle('DDIM Inversion（上）と Sampling（下）の過程\n上: 画像→ノイズ, 下: ノイズ→画像',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_310_03_process_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 上段: 入力画像が徐々にノイズに変換される（Inversion）")
print("   下段: ノイズから元の画像が復元される（Sampling）")
print("   → この往復が精度良くできることが、画像編集の鍵")

In [ ]:
# ============================================================
# ノイズ空間での簡易編集
# Inversion後のノイズを操作して再生成
# ============================================================

# 2つの画像のInversionノイズを補間
x_0_pair = x_0[:2]  # 2枚の画像
x_T_pair, _ = ddim_inversion(noise_pred, x_0_pair, schedule, n_inference_steps=50)

n_interp = 10
alphas = np.linspace(0, 1, n_interp)

fig, axes = plt.subplots(1, n_interp, figsize=(16, 2))

for i, alpha in enumerate(alphas):
    x_T_interp = (1 - alpha) * x_T_pair[0:1] + alpha * x_T_pair[1:2]
    x_result, _ = ddim_sample(noise_pred, x_T_interp, schedule, n_inference_steps=50)
    img = (x_result[0].cpu().numpy().reshape(28, 28) + 1) / 2
    axes[i].imshow(np.clip(img, 0, 1), cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'α={alpha:.1f}', fontsize=9)

fig.suptitle('ノイズ空間での補間（DDIM Inversion後）\n2つの画像のInversionノイズを線形補間してSampling',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_310_04_noise_interpolation.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Inversionノイズの補間 = 元画像の意味的な変容")
print("   GANの潜在空間補間と同様の効果を、拡散モデルで実現")

---

## まとめ

### 🎯 このノートブックで学んだこと

**DDPMとDDIMの違い**
- ✓ DDPM: 確率的（毎回異なる結果）→ Inversion不可能
- ✓ DDIM: 決定的（η=0で同一結果）→ Inversion可能

**DDIM Inversionの原理**
- ✓ DDIMサンプリング式を逆方向に適用：画像→ノイズ
- ✓ ほぼ完璧な再構成が可能（往復精度が高い）
- ✓ GAN Inversionよりも高速・高精度

**画像編集パイプライン**
- ✓ Inversion → ノイズ空間での操作 → Sampling
- ✓ ノイズ補間でGAN潜在空間と同様の変容が可能

### 📊 チートシート

| 用語 | 意味 |
|------|------|
| DDIM Inversion | 画像→ノイズへの決定的逆変換 |
| αbar (ᾱₜ) | ノイズスケジュールの累積積 |
| η = 0 | DDIMの決定的モード |
| Trajectory | Inversion/Samplingの中間状態列 |

### ⚠️ よくある間違い

❌ 「Inversionステップ数は少ないほど速くて良い」
✅ ステップ数が少ないとInversionの精度が下がり、再構成品質が低下する。50-100ステップが一般的

❌ 「DDIMは常にDDPMより良い」
✅ 画像生成品質ではDDPMのほうが優れる場合もある。DDIMの利点は「決定的であること」と「Inversionが可能なこと」

### ✅ 学習チェックリスト

- [ ] DDIMのサンプリング式をDDPMとの違いとともに書けるか？
- [ ] DDIM Inversionの式の導出を概説できるか？
- [ ] Inversionが画像編集にどう貢献するか説明できるか？

---

**次のステップ**: ノートブック311「ノイズ空間補間」で、Inversion後のノイズ空間で高品質な画像変容を実現する技術を学びます！

---

## 🎓 自己評価クイズ

### Q1: DDIMが「決定的」であることがInversionに不可欠な理由は？

<details>
<summary>💡 答えを見る</summary>

**答え**: 決定的な写像でなければ、逆写像も一意に定まらないため

DDPMの逆過程にはランダムノイズzが含まれるため、同じxₜから異なるx₀が生成される。
これでは「どのxₜがこのx₀に対応するか」が定まらない。
DDIMはη=0でランダム項を除去し、一意な写像にすることでInversionを可能にする。

</details>

---

### Q2: DDIM InversionのMSEが完全にゼロにならない理由は？

<details>
<summary>💡 答えを見る</summary>

**答え**: 離散化誤差とネットワークの予測誤差のため

DDIMの式は連続時間では完璧だが、実際は有限のタイムステップで近似するため離散化誤差が生じる。
また、ノイズ予測ネットワークε_θの予測が完璧でないことも誤差の要因。ステップ数を増やすと改善する。

</details>

---

### Q3: DDIM Inversionで得られたノイズxₜを2つの画像間で線形補間すると何が起きるか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 2つの画像の間の意味的な変容が生じる

Inversionノイズはその画像の「情報を保持したノイズ」なので、それらを補間してSamplingすると、2つの画像の特徴が混ざり合った画像が生成される。これはGANの潜在空間補間と類似した効果。

</details>